# Phase 2 Training - Upload: train_v2.pkl, val_v2.pkl, test_v2.pkl, best_model_full.pt

In [ ]:
import os
files = ['train_v2.pkl', 'val_v2.pkl', 'test_v2.pkl', 'best_model_full.pt']
for f in files: print(f, 'EXISTS' if os.path.exists(f) else 'MISSING')

In [ ]:
!pip install transformers torch scikit-learn pandas numpy tqdm -q
print('Installed')

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
import pickle
with open('train_v2.pkl', 'rb') as f: train_data = pickle.load(f)
with open('val_v2.pkl', 'rb') as f: val_data = pickle.load(f)
with open('test_v2.pkl', 'rb') as f: test_data = pickle.load(f)
print('Train:', len(train_data['labels']))
print('Graph dims:', train_data['graph_features'].shape)

In [ ]:
from torch.utils.data import Dataset
from transformers import DistilBertTokenizer
import torch

class JobDataset(Dataset):
    def __init__(self, texts, meta, graph, labels):
        self.texts = texts
        self.meta = torch.tensor(meta, dtype=torch.float32)
        self.graph = torch.tensor(graph, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=512, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(),
                'meta': self.meta[idx], 'graph': self.graph[idx], 'label': self.labels[idx]}

train_ds = JobDataset(train_data['texts'], train_data['metadata'], train_data['graph_features'], train_data['labels'])
val_ds = JobDataset(val_data['texts'], val_data['metadata'], val_data['graph_features'], val_data['labels'])

In [ ]:
import torch.nn as nn
from transformers import DistilBertModel

class ModelV2(nn.Module):
    def __init__(self, n_meta=41, n_graph=14):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        for p in self.bert.transformer.layer[:4].parameters(): p.requires_grad = False
        self.meta_enc = nn.Sequential(nn.Linear(n_meta, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
                                      nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2))
        self.graph_enc = nn.Sequential(nn.Linear(n_graph, 64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(0.3),
                                       nn.Linear(64, 48), nn.LayerNorm(48), nn.ReLU(), nn.Dropout(0.2), nn.Linear(48, 32), nn.ReLU())
        self.fusion = nn.Sequential(nn.Linear(768+64+32, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(0.4),
                                    nn.Linear(512, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))
    def forward(self, ids, mask, meta, graph):
        text = self.bert(ids, mask).last_hidden_state[:, 0, :]
        return self.fusion(torch.cat([text, self.meta_enc(meta), self.graph_enc(graph)], dim=1)).squeeze(-1)

model = ModelV2()

In [ ]:
ckpt = torch.load('best_model_full.pt', map_location='cpu', weights_only=False)
state = ckpt.get('model_state_dict', ckpt)
model_dict = model.state_dict()
filtered = {k: v for k, v in state.items() if k in model_dict and model_dict[k].shape == v.shape}
model.load_state_dict(filtered, strict=False)
print(f'Loaded {len(filtered)}/{len(model_dict)} layers')

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
import torch.optim as optim
from tqdm import tqdm
import numpy as np

BATCH, EPOCHS = 16, 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH)
bert_params = list(model.bert.named_parameters())
other_params = list(model.meta_enc.parameters()) + list(model.graph_enc.parameters()) + list(model.fusion.parameters())
optimizer = optim.AdamW([{'params': [p for n, p in bert_params], 'lr': 2e-5},
                         {'params': other_params, 'lr': 2e-4}])
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device))
print('Ready on', device)

In [ ]:
best_auc, patience = 0, 0
for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    model.train()
    train_loss, train_preds, train_labels = 0, [], []
    for batch in tqdm(train_loader, desc='Train'):
        optimizer.zero_grad()
        logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device),
                      batch['meta'].to(device), batch['graph'].to(device))
        loss = criterion(logits, batch['label'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        train_preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
        train_labels.extend(batch['label'].cpu().numpy())
    train_auc = roc_auc_score(train_labels, train_preds)
    print(f'Train Loss: {train_loss/len(train_loader):.4f}, AUC: {train_auc:.4f}')
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Val'):
            val_preds.extend(torch.sigmoid(model(batch['input_ids'].to(device), batch['attention_mask'].to(device),
                           batch['meta'].to(device), batch['graph'].to(device))).cpu().numpy())
            val_labels.extend(batch['label'].numpy())
    val_auc = roc_auc_score(val_labels, val_preds)
    print(f'Val AUC: {val_auc:.4f}')
    if val_auc > best_auc:
        best_auc = val_auc
        patience = 0
        torch.save({'model': model.state_dict(), 'auc': val_auc}, 'phase2_best.pt')
        print(f'Saved! Best: {best_auc:.4f}')
    else:
        patience += 1
        if patience >= 3: print('Early stop'); break
print(f'Done! Best AUC: {best_auc:.4f}')

In [ ]:
from google.colab import files
files.download('phase2_best.pt')
print('Downloaded phase2_best.pt!')